In [1]:
import time
import pandas as pd
import numpy as np
from pmdarima import auto_arima
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

In [2]:
data = pd.read_csv("../../../data/raw/kathmandu_full_raw_2023_2024.csv")

data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time").reset_index(drop=True)
data = data.set_index("time")
data.index.freq = "h"

In [3]:
# --- Split ---
split = int(np.ceil(0.8 * len(data)))
train = data.iloc[:split]
test  = data.iloc[split:]

In [4]:
# --- Train ---
train_start = time.time()
model = auto_arima(train["pm2_5"], seasonal=False, stepwise=True,
                   information_criterion="aic")
train_time = time.time() - train_start

In [5]:
# --- Predict ---
train_preds = model.predict_in_sample()

inference_start = time.time()
test_preds = model.predict(n_periods=len(test))
inference_time = (time.time() - inference_start) / len(test)

In [6]:
# --- Metrics ---
train_rmse = root_mean_squared_error(train["pm2_5"], train_preds)
train_mae  = mean_absolute_error(train["pm2_5"], train_preds)
train_r2   = r2_score(train["pm2_5"], train_preds)

test_rmse = root_mean_squared_error(test["pm2_5"], test_preds)
test_mae  = mean_absolute_error(test["pm2_5"], test_preds)
test_r2   = r2_score(test["pm2_5"], test_preds)

print(f"Train RMSE: {train_rmse:.4f} | MAE: {train_mae:.4f} | R2: {train_r2:.4f}")
print(f"Test  RMSE: {test_rmse:.4f} | MAE: {test_mae:.4f} | R2: {test_r2:.4f}")
print(f"Training Time:  {train_time:.2f}s")
print(f"Inference Time: {inference_time:.6f}s per sample")

Train RMSE: 3.8305 | MAE: 2.1735 | R2: 0.9370
Test  RMSE: 40.3486 | MAE: 27.5013 | R2: -0.8216
Training Time:  769.37s
Inference Time: 0.000039s per sample
